In [ ]:
# SCRIPT PARA CRIAÇÃO DO BANCO DE DADOS VETORIAL
# UTILIZANDO OLLAMA PARA RODAR O MODELO DE EMBEDDING LOCALMENTE (QWEN3-EMBEDDING)
# VERSÃO NOVA, COM UNSTRUCTED PDF

import os
import ollama 
import chromadb
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm
import time
from unstructured.partition.html import partition_html
import re

# --- 1. CONFIGURAÇÕES ---
PDF_DIRECTORY = "datasheets_html"
DB_PATH = "vector_db_v3"
COLLECTION_NAME = "rag_datasheets_v3"
EMBEDDING_MODEL = "qwen3-embedding"
CHUNK_SIZE = 768
CHUNK_OVERLAP = 120

# --- CONFIGURAÇÕES DO LOTE ---
PDF_BATCH_SIZE = 168  # NÚMERO DE PDFs A PROCESSAR POR EXECUÇÃO
PROCESSED_LOG_FILE = "processed_pdfs_v3.txt" # ARQUIVO DE LOG PARA CONTROLAR O QUE JÁ FOI PROCESSADO


# --- Funções ---

def clean_extracted_text(text: str) -> str:
    """
    Limpa e normaliza o texto extraído, removendo ruídos de PDF e corrigindo quebras de linha.
    """
    # Remove cabeçalhos/rodapés comuns (ex: © 2008 Microchip Technology Inc. PIC18F8722 FAMILY)
    text = re.sub(r'© \d{4}.*', '', text)
    text = re.sub(r'DS\d+.*', '', text)
    re
    # Remove caracteres de erro de decodificação como (cid:129)
    text = re.sub(r'\(cid:\d+\)', '', text)
    
    # --- LÓGICA DE CORREÇÃO DE QUEBRA DE LINHA ---
    # 1. Usa um marcador temporário para parágrafos reais (2+ quebras de linha)
    text = re.sub(r'(\n\s*){2,}', '<<PARAGRAPH_BREAK>>', text)
    # 2. Substitui quebras de linha únicas (provavelmente indesejadas) por um espaço
    text = text.replace('\n', ' ')
    # 3. Restaura os parágrafos reais
    text = text.replace('<<PARAGRAPH_BREAK>>', '\n\n')
    
    # Remove espaços múltiplos que podem ter sido criados
    text = re.sub(r' {2,}', ' ', text)
    
    return text.strip()

def get_processed_files(log_file_path):
    """Lê o arquivo de log e retorna um conjunto de nomes de arquivos já processados."""
    if not os.path.exists(log_file_path):
        return set()
    with open(log_file_path, 'r') as f:
        # Usa .strip() para remover quebras de linha e espaços em branco
        return set(line.strip() for line in f)

def load_and_partition_html_unstructured(html_filepaths):
    """
    Carrega HTMLs com unstructured
    """
    documents = []
    print("Extraindo e particionando elementos dos HTMLs com Unstructured.io...")
    
    for filepath in tqdm(html_filepaths, desc="Processando HTMLs com Unstructured"):
        try:
            # Substitui partition_pdf por partition_html
            # Remove 'strategy' e 'model_name', pois não se aplicam ao HTML
            elements = partition_html(
                filename=filepath,
                infer_table_structure=True
            )
            
            # O restante da lógica para unir e limpar o texto permanece o mesmo
            raw_text = "\n".join([el.text for el in elements])
            
            # Aplica a função de limpeza para normalizar o texto
            cleaned_text = clean_extracted_text(raw_text)

            if cleaned_text:
                filename = os.path.basename(filepath)
                documents.append((cleaned_text, filename))

        except Exception as e:
            filename = os.path.basename(filepath)
            print(f"\nAVISO: Falha ao processar o arquivo '{filename}'. Pulando este arquivo.")
            print(f"   -> Motivo: {e}\n")
            
    return documents

def split_documents(documents, chunk_size, chunk_overlap):
    """
    Divide os documentos em chunks e extrai metadados dos modelos de microcontroladores.
    """
    print("Dividindo os documentos em chunks e extraindo metadados...")
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""],
    )
    
    all_chunks = []
    for text, source in tqdm(documents, desc="Dividindo e analisando chunks"):
        chunks = text_splitter.split_text(text)
        
        for chunk_text in chunks:
            models_found = re.findall(r'\b(PIC|AT\w*|STM32|LPC)\d+\w*\b', chunk_text, re.IGNORECASE)
            unique_models = list(set(models_found))
            models_str = ", ".join(unique_models)

            all_chunks.append({
                "text": chunk_text,
                "metadata": {
                    "source": source,
                    "models": models_str
                }
            })
            
    return all_chunks

def get_ollama_embeddings_batch(texts, model):
    # Esta função permanece a mesma
    embeddings_list = []
    for text in texts:
        try:
            response = ollama.embeddings(model=model, prompt=text)
            embeddings_list.append(response['embedding'])
        except Exception as e:
            print(f"Erro ao gerar embedding para um texto: {e}")
            embeddings_list.append(None) 
    return [emb for emb in embeddings_list if emb is not None]


def criar_banco_vetorial_em_lotes():
    print("--- Iniciando o processo de criação do Banco de Dados Vetorial (Modo Lote) ---")

    # 1. Identificar arquivos já processados
    processed_files = get_processed_files(PROCESSED_LOG_FILE)
    print(f"Encontrados {len(processed_files)} arquivos já processados no log.")

    # 2. Identificar todos os arquivos HTML disponíveis e filtrar os que faltam processar
    # CORREÇÃO AQUI: Alterado de ".pdf" para ".html" e ".htm"
    all_html_files = sorted([
        f for f in os.listdir(PDF_DIRECTORY) 
        if f.lower().endswith((".html", ".htm"))
    ])
    
    # E AQUI: Usar a nova variável
    files_to_process = [f for f in all_html_files if f not in processed_files]
    
    # 3. Verificar se há trabalho a ser feito
    if not files_to_process:
        # AQUI: Mensagem corrigida para HTML
        print("\nTodos os arquivos HTML já foram processados. Nenhuma ação necessária.")
        return

    # 4. Selecionar o próximo lote de arquivos para processar
    batch_filenames = files_to_process[:PDF_BATCH_SIZE]
    batch_filepaths = [os.path.join(PDF_DIRECTORY, fname) for fname in batch_filenames]
    
    print(f"\nTotal de arquivos restantes: {len(files_to_process)}")
    print(f"Processando um lote de {len(batch_filenames)} arquivos:")
    for fname in batch_filenames:
        print(f"  - {fname}")
    
    # 5. Processar o lote (carregar, dividir, embedar, adicionar ao DB)
    # Esta chamada agora funcionará, pois batch_filepaths não estará vazio
    documents = load_and_partition_html_unstructured(batch_filepaths)

    chunks_with_metadata = split_documents(documents, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
    if not chunks_with_metadata: return
        
    print(f"\nTotal de {len(documents)} documentos processados neste lote.")
    print(f"Total de {len(chunks_with_metadata)} chunks gerados.")

    client_chroma = chromadb.PersistentClient(path=DB_PATH)
    collection = client_chroma.get_or_create_collection(name=COLLECTION_NAME, metadata={"hnsw:space": "cosine"})

    print(f"Gerando embeddings com o modelo local '{EMBEDDING_MODEL}' e salvando no ChromaDB...")
    batch_size = 100 
    total_batches = (len(chunks_with_metadata) + batch_size - 1) // batch_size

    for i in tqdm(range(0, len(chunks_with_metadata), batch_size), desc="Adicionando ao ChromaDB", total=total_batches):
        batch = chunks_with_metadata[i:i + batch_size]
        texts_to_embed = [item['text'] for item in batch]
        metadatas = [item['metadata'] for item in batch]
        
        embeddings = get_ollama_embeddings_batch(texts_to_embed, EMBEDDING_MODEL)
        
        # Lógica de segurança para o caso de algum embedding falhar
        if len(embeddings) < len(texts_to_embed):
            print("Aviso: Houve falha em alguns embeddings. Sincronizando dados antes de adicionar.")
            # Esta é uma maneira mais eficiente de sincronizar
            successful_indices = [i for i, emb in enumerate(get_ollama_embeddings_batch(texts_to_embed, EMBEDDING_MODEL)) if emb is not None]
            texts_to_embed = [texts_to_embed[i] for i in successful_indices]
            metadatas = [metadatas[i] for i in successful_indices]
            embeddings = [emb for emb in get_ollama_embeddings_batch(texts_to_embed, EMBEDDING_MODEL) if emb is not None] # Re-calcula para garantir
            
        ids = [f"chunk_{int(time.time() * 1000)}_{i+j}" for j in range(len(embeddings))] # ID mais robusto para evitar colisões
        
        if embeddings:
            collection.add(embeddings=embeddings, documents=texts_to_embed, metadatas=metadatas, ids=ids)

    # 6. Atualizar o arquivo de log com os nomes dos arquivos processados com sucesso
    try:
        with open(PROCESSED_LOG_FILE, 'a') as f:
            for filename in batch_filenames:
                f.write(f"{filename}\n")
        print(f"\nLog '{PROCESSED_LOG_FILE}' atualizado com {len(batch_filenames)} novos arquivos.")
    except Exception as e:
        print(f"ERRO CRÍTICO: Não foi possível atualizar o arquivo de log: {e}")


    count = collection.count()
    print("\n--- Processo do Lote Concluído com Sucesso! ---")
    print(f"Total de chunks (vetores) na coleção agora: {count}")

# Executa a função de processamento em lotes
criar_banco_vetorial_em_lotes()

In [ ]:
import os
import pandas as pd
import json
import ollama
import chromadb
from openai import OpenAI
from tqdm import tqdm
import textwrap
import re

#  1. CONFIGURAÇÕES GERAIS
NVAPI_KEY = "SUA_API_AQUI"
DB_PATH = "vector_db_v3"
COLLECTION_NAME = "rag_datasheets_v3"
EMBEDDING_MODEL = "qwen3-embedding"
NVIDIA_MODEL = "meta/llama-4-scout-17b-16e-instruct"
NUM_RETRIEVED_DOCS = 5 # QUANTIDADE CHUNKS A SEREM RECUPERADOS

# 2. INICIALIZAÇÃO DOS CLIENTES 
print("--- Iniciando o Processo de Inferência com RAG ---")
print("Usando embeddings do Ollama e inferência da NVIDIA.")

try:
    client_nvidia = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=NVAPI_KEY)
    client_chroma = chromadb.PersistentClient(path=DB_PATH)
    collection = client_chroma.get_collection(name=COLLECTION_NAME)
    print(f"Clientes inicializados. Conectado à coleção '{COLLECTION_NAME}' com {collection.count()} documentos.")
except Exception as e:
    print(f"Erro ao inicializar clientes: {e}")
    raise

# 3. FUNÇÕES DO WORKFLOW RAG

def extract_model_from_query(query: str) -> str | None:
    # Padrão regex aprimorado para pegar os modelos
    match = re.search(r'\b(PIC|AT\w*|STM32|LPC)\d+\w*\b', query, re.IGNORECASE)
    return match.group(0) if match else None

def embed_query(query_text): 
# Gera o embedding para uma única pergunta usando Ollama
    try:
        response = ollama.embeddings(
            model=EMBEDDING_MODEL,
            prompt=query_text
        )
        return response['embedding']
    except Exception as e:
        print(f"Erro ao gerar embedding com Ollama: {e}")
        return None

def retrieve_context(query_embedding, query_text, n_results=NUM_RETRIEVED_DOCS):
    """
    Recupera um número maior de chunks e depois filtra em Python
    para encontrar os mais relevantes para o modelo especificado.
    """
    model_in_query = extract_model_from_query(query_text)

    # Se não houver modelo na pergunta, faz a busca normal

    if not model_in_query:
        print("   Nenhum modelo específico detectado. Buscando em todo o banco de dados.")
        results = collection.query(query_embeddings=[query_embedding], n_results=n_results)
        return results['documents'][0] if results and results['documents'] else []

    # --- LÓGICA DE PÓS-FILTRAGEM ---
    print(f"   Modelo '{model_in_query}' detectado. Iniciando busca com pós-filtragem.")
    
    NUMERO_CANDIDATOS = 15
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=NUMERO_CANDIDATOS,
        include=["metadatas", "documents"]
    )

    if not results or not results['documents']:
        return []

    # 2. Filtra os resultados em Python
    filtered_documents = []
    
    for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
        
        # >>> ALTERAÇÃO AQUI <<<
        # Verifica primeiro se 'meta' não é None e se a chave 'models' existe
        if meta and 'models' in meta:
            # Pega o valor do metadado 'models'
            models_metadata = meta['models']
            
            # Garante que o valor é uma string antes de usar o operador 'in'
            if isinstance(models_metadata, str) and model_in_query in models_metadata:
                filtered_documents.append(doc)
        
        # Para de procurar quando já tivermos chunks suficientes
        if len(filtered_documents) >= n_results:
            break
            
    if not filtered_documents:
            print(f"   AVISO: Nenhum chunk encontrado para o modelo '{model_in_query}' após a filtragem.")
            # Como fallback, retorna os resultados da busca original sem filtro
            return results['documents'][0][:n_results]

    return filtered_documents
    

def construct_rag_prompt(context_list, question):
    """Constrói o prompt final para o LLM."""
    context_str = "\n\n".join(context_list)
    prompt = (
        "Responda a pergunta \n"
        "Utilize as informações fornecidas no contexto para formular sua resposta."
        "\n\n--- CONTEXTO ---\n"
        f"{context_str}"
        "\n\n--- PERGUNTA ---\n"
        f"{question}"
    )
    return prompt

def generate_rag_response_nvidia(prompt):
    """Envia o prompt para a API da NVIDIA."""
    try:
        completion = client_nvidia.chat.completions.create(
            model=NVIDIA_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
            max_tokens=512,
            stream=False
        )
        return completion.choices[0].message.content.strip() if completion.choices else "Erro: Resposta vazia"
    except Exception as e:
        print(f"ERRO CRÍTICO na API da NVIDIA: {e}")
        return "Erro ao chamar API"


def testar_pergunta_manual():
    # FUNÇÃO DE TESTE MANUAL

    # ##################################################################
    # PERGUNTA DE TESTE AQUI
    # ##################################################################
    pergunta_teste = "Qual a velocidade (clock) do microcontrolador ATMEGA2560-16AUR?"
    
    print("="*80)
    print(f"PERGUNTA DE TESTE: {pergunta_teste}\n")

    # >>> CORREÇÃO CRÍTICA AQUI <<<
    # A geração do embedding DEVE acontecer ANTES da recuperação do contexto.
    
    # 1. Gerar embedding para a pergunta
    print(f"1. Gerando embedding para a pergunta com o modelo local '{EMBEDDING_MODEL}'...")
    query_embedding = embed_query(pergunta_teste)
    
    if query_embedding is None:
        print("   Falha ao gerar embedding. Abortando.")
        return
        
    print("   Embedding gerado com sucesso.")

    # 2. Recuperar contexto do ChromaDB
    print("\n2. Recuperando contexto do banco de dados vetorial...")
    # Agora esta linha funcionará, pois 'query_embedding' já existe.
    contexto_recuperado = retrieve_context(query_embedding, pergunta_teste)
    
    if not contexto_recuperado:
        print("   AVISO: Nenhum contexto relevante foi encontrado.")
        # Mesmo sem contexto, podemos continuar para ver o que o LLM responde sem ajuda.
        # Se preferir parar, pode adicionar um 'return' aqui.
    
    print(f"   {len(contexto_recuperado)} chunks de contexto recuperados:")
    print("-" * 50)
    for i, chunk in enumerate(contexto_recuperado):
        print(f"Chunk {i+1}:\n{textwrap.fill(chunk, width=100)}\n")
    print("-" * 50)

    # 3. Construir o prompt RAG
    print("\n3. Construindo o prompt para o modelo...")
    rag_prompt = construct_rag_prompt(contexto_recuperado, pergunta_teste)
    # Descomente a linha abaixo se quiser ver o prompt completo enviado ao modelo
    # print(f"\n--- PROMPT COMPLETO ---\n{rag_prompt}\n-----------------------\n")

    # 4. Gerar resposta com o modelo da NVIDIA
    print("\n4. Enviando prompt para a API da NVIDIA e aguardando a resposta...")
    resposta_rag = generate_rag_response_nvidia(rag_prompt)
    
    # 5. Exibir resultado final
    print("\n" + "="*80)
    print("RESULTADO FINAL DO TESTE")
    print("="*80)
    print(f">> Pergunta: {pergunta_teste}")
    print("\n>> Resposta do Modelo (RAG):")
    print(textwrap.fill(resposta_rag, width=100))
    print("="*80)

# --- 5. EXECUÇÃO DO TESTE ---
# Chama a função de teste
testar_pergunta_manual()

In [ ]:
### NVIDIA

import os
import pandas as pd
import json
import ollama
import chromadb
from openai import OpenAI
from tqdm import tqdm
import textwrap
import re
import time

# 1. CONFIGURAÇÕES GERAIS
NVAPI_KEY = "SUA_API_AQUI"
DB_PATH = "vector_db_v3"
COLLECTION_NAME = "rag_datasheets_v3"
EMBEDDING_MODEL = "qwen3-embedding"
NVIDIA_MODEL = "microsoft/phi-4-mini-instruct"
NUM_RETRIEVED_DOCS = 5 # QUANTIDADE CHUNKS A SEREM RECUPERADOS

# --- NOVAS CONFIGURAÇÕES ---
# Nome do arquivo de entrada (o CSV que você anexou)
ARQUIVO_ENTRADA = "microcontroladores-populares.xlsx"
# Nome do arquivo JSON de saída
ARQUIVO_SAIDA = "resultados_rag_phi-4-mini_v3.json"

# Variantes do prompt: True = com instrução ("Se não souber a resposta, retorne 'Não sei'."), False = sem instrução
VARIANTES_INSTRUCAO = [True, False]

# --- TRATAMENTO DE ERROS DE API ---
# Chamadas que falham são repetidas com espera crescente. Se todas as tentativas falharem, a resposta é
# gravada com o prefixo ERRO_API: ela não é enviada para extração, é contada à parte na avaliação
# e é refeita automaticamente na próxima execução do script.
ERRO_API = "erro_api"
TENTATIVAS_API = 3
ESPERA_INICIAL_S = 5  # dobra a cada nova tentativa (5s, 10s, ...)

def eh_erro_api(texto):
    return str(texto).startswith(ERRO_API)

def chamar_com_retentativas(chamada, descricao):
    """Executa 'chamada' (função sem argumentos que retorna o texto da resposta ou levanta exceção).
    Retorna o texto, ou f"{ERRO_API}: <motivo>" se todas as tentativas falharem."""
    for tentativa in range(1, TENTATIVAS_API + 1):
        try:
            return chamada()
        except Exception as e:
            print(f"   [{descricao}] Falha na tentativa {tentativa}/{TENTATIVAS_API}: {e}")
            if tentativa < TENTATIVAS_API:
                time.sleep(ESPERA_INICIAL_S * 2 ** (tentativa - 1))
            else:
                return f"{ERRO_API}: {e}"

# Mapeamento das colunas de pergunta para as colunas de resposta (ground truth)
MAPEAMENTO_PERGUNTAS = {
    "PERGUNTA01": "Flash Memory",
    "PERGUNTA02": "Speed",
    "PERGUNTA03": "Number of I/O",
    "PERGUNTA04": "CommCAN",
    "PERGUNTA05": "CommI2C",
    "PERGUNTA06": "CommEthernet"
}
# --- FIM DAS NOVAS CONFIGURAÇÕES ---


# 2. INICIALIZAÇÃO DOS CLIENTES 
print("--- Iniciando o Processo de Inferência com RAG ---")
print("Usando embeddings do Ollama e inferência da NVIDIA.")

try:
    client_nvidia = OpenAI(
        base_url="https://integrate.api.nvidia.com/v1", 
        api_key=NVAPI_KEY,
        timeout=60.0  # Adiciona timeout de 60s para evitar travamentos
    )
    client_chroma = chromadb.PersistentClient(path=DB_PATH)
    collection = client_chroma.get_collection(name=COLLECTION_NAME)
    print(f"Clientes inicializados. Conectado à coleção '{COLLECTION_NAME}' com {collection.count()} documentos.")
except Exception as e:
    print(f"Erro ao inicializar clientes: {e}")
    raise

# 3. FUNÇÕES DO WORKFLOW RAG

def extract_model_from_query(query: str) -> str | None:
    match = re.search(r'\b(PIC|AT\w*|STM32|LPC)\d+\w*\b', query, re.IGNORECASE)
    return match.group(0) if match else None

def embed_query(query_text): 
    try:
        response = ollama.embeddings(
            model=EMBEDDING_MODEL,
            prompt=query_text
        )
        return response['embedding']
    except Exception as e:
        print(f"Erro ao gerar embedding com Ollama: {e}")
        return None

def deduplicate_results(results_list: list[dict]) -> list[dict]:
    """
    Recebe uma lista de dicionários e remove duplicatas com base no 'chunk_content'.
    """
    seen_content = set()
    deduplicated_list = []
    for item in results_list:
        content = item['chunk_content']
        if content not in seen_content:
            seen_content.add(content)
            deduplicated_list.append(item)
    return deduplicated_list

# >>> MODIFICADA <<<
def retrieve_context(query_embedding, query_text, n_results=NUM_RETRIEVED_DOCS):
    """
    Recupera chunks, de-duplica e filtra.
    Retorna uma lista de dicionários: [{'chunk_content': '...', 'source_file': '...'}, ...]
    """
    model_in_query = extract_model_from_query(query_text)

    # --- Lógica de busca base ---
    query_params = {
        "query_embeddings": [query_embedding],
        # Aumentamos o N de candidatos para ter mais chance de achar únicos
        "n_results": 20, 
        "include": ["metadatas", "documents"]
    }

    if not model_in_query:
        print("   Nenhum modelo específico detectado. Buscando em todo o banco de dados.")
        try:
            results = collection.query(**query_params)
        except Exception as e:
            print(f"   ERRO ao consultar o ChromaDB: {e}")
            return []
            
        if not results or not results['documents']:
            return []
        
        # Empacota os resultados
        packaged_results = []
        for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
            source_filename = meta.get('source', 'Fonte Desconhecida')
            packaged_results.append({
                "chunk_content": doc,
                "source_file": source_filename
            })
        
        # <<< APLICA A ESTRATÉGIA AQUI >>>
        deduplicated_results = deduplicate_results(packaged_results)
        
        print(f"   Recuperados {len(packaged_results)} chunks, {len(deduplicated_results)} são únicos.")
        # Retorna apenas o N desejado de chunks únicos
        return deduplicated_results[:n_results]

    # --- LÓGICA DE PÓS-FILTRAGEM ---
    print(f"   Modelo '{model_in_query}' detectado. Iniciando busca com pós-filtragem.")
    
    try:
        results = collection.query(**query_params)
    except Exception as e:
        print(f"   ERRO ao consultar o ChromaDB: {e}")
        return []

    if not results or not results['documents']:
        return []

    # Filtra os resultados em Python
    filtered_documents_with_meta = []
    for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
        if meta and 'models' in meta:
            models_metadata = meta['models']
            if isinstance(models_metadata, str) and model_in_query in models_metadata:
                source_filename = meta.get('source', 'Fonte Desconhecida')
                filtered_documents_with_meta.append({
                    "chunk_content": doc,
                    "source_file": source_filename
                })
    
    # <<< APLICA A ESTRATÉGIA AQUI >>>
    deduplicated_results = deduplicate_results(filtered_documents_with_meta)
    print(f"   Filtrados {len(filtered_documents_with_meta)} chunks, {len(deduplicated_results)} são únicos.")

    # Se a filtragem não encontrar nada, usa o fallback
    if not deduplicated_results:
        print(f"   AVISO: Nenhum chunk único encontrado para '{model_in_query}'. Usando fallback.")
        
        fallback_documents = []
        for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
            source_filename = meta.get('source', 'Fonte Desconhecida')
            fallback_documents.append({
                "chunk_content": doc,
                "source_file": source_filename
            })
        
        # De-duplica o fallback também
        deduplicated_fallback = deduplicate_results(fallback_documents)
        return deduplicated_fallback[:n_results]

    return deduplicated_results[:n_results]

def construct_rag_prompt(context_list_of_dicts, question, usar_instrucao=True):
    """Constrói o prompt final para o LLM a partir da lista de dicionários."""
    
    # Extrai apenas o conteúdo do chunk para o prompt
    chunk_contents = [item['chunk_content'] for item in context_list_of_dicts]
    context_str = "\n\n".join(chunk_contents)
    
    instrucao = " Se não souber a resposta, retorne 'Não sei'." if usar_instrucao else ""
    prompt = (
        "Responda a pergunta \n"
        "Utilize as informações fornecidas no contexto para formular sua resposta."
        f"{instrucao}"
        "\n\n--- CONTEXTO ---\n"
        f"{context_str}"
        "\n\n--- PERGUNTA ---\n"
        f"{question}"
    )
    return prompt

def generate_rag_response_nvidia(prompt):
    """Envia o prompt para a API da NVIDIA."""
    def chamada():
        print("   Enviando para a API NVIDIA...")
        completion = client_nvidia.chat.completions.create(
            model=NVIDIA_MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.2,
            max_tokens=512,
            stream=False
        )
        print("   Resposta recebida.")

        if not completion.choices:
            raise ValueError("a API não retornou 'choices'")

        # Pega o objeto 'message' da primeira escolha
        message = completion.choices[0].message

        # Tenta obter a resposta do campo 'content' primeiro
        content = message.content
        if content and content.strip():
            print("   [INFO] Resposta encontrada no campo 'content'.")
            return content.strip()

        # Se 'content' estiver vazio, tenta obter do 'reasoning_content'
        reasoning_content = getattr(message, 'reasoning_content', None)
        if reasoning_content and reasoning_content.strip():
            print("   [AVISO] 'content' estava vazio. Usando 'reasoning_content' como resposta.")
            return reasoning_content.strip()

        # Se ambos estiverem vazios
        raise ValueError("'content' e 'reasoning_content' estão vazios")

    return chamar_com_retentativas(chamada, "NVIDIA")


def salvar_json(data, filename):
    """Função auxiliar para salvar o JSON."""
    try:
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
        print(f"\n[Progresso salvo em '{filename}']")
    except Exception as e:
        print(f"\n[ERRO AO SALVAR JSON: {e}]")

def processar_planilha():
    todos_os_resultados = []
    
    try:
        if os.path.exists(ARQUIVO_SAIDA):
            with open(ARQUIVO_SAIDA, 'r', encoding='utf-8') as f:
                todos_os_resultados = json.load(f)
            print(f"Carregados {len(todos_os_resultados)} resultados de execuções anteriores.")
    except Exception as e:
        print(f"Não foi possível carregar resultados anteriores: {e}")
        todos_os_resultados = []

    # Perguntas com erro de API são descartadas para serem refeitas nesta execução
    validos = [r for r in todos_os_resultados
               if not any(eh_erro_api(m.get("resposta_completa")) for m in r.get("respostas_modelos", []))]
    if len(validos) < len(todos_os_resultados):
        print(f"{len(todos_os_resultados) - len(validos)} perguntas com erro de API serão refeitas.")
    todos_os_resultados = validos

    # Perguntas já respondidas em execuções anteriores (permite retomar sem duplicar resultados)
    ja_processadas = {(r.get("linha_excel"), r.get("coluna_pergunta")) for r in todos_os_resultados}

    try:
        df = pd.read_excel(ARQUIVO_ENTRADA)
        print(f"Arquivo '{ARQUIVO_ENTRADA}' carregado. {len(df)} linhas para processar.")
    except FileNotFoundError:
        print(f"ERRO: Arquivo de entrada não encontrado: {ARQUIVO_ENTRADA}")
        return
    except Exception as e:
        print(f"Erro ao ler o arquivo CSV: {e}")
        return

    try:
        print("Iniciando processamento das perguntas...")
        for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Processando Linhas"):
            
            for coluna_pergunta, coluna_resposta in MAPEAMENTO_PERGUNTAS.items():
                
                pergunta_texto = row[coluna_pergunta]
                resposta_verdadeira = row[coluna_resposta]
                
                if not isinstance(pergunta_texto, str) or pd.isna(pergunta_texto):
                    continue

                if (int(index), coluna_pergunta) in ja_processadas:
                    continue

                print("\n" + "="*80)
                print(f"Processando: [Linha {index + 2}] - [{coluna_pergunta}]")
                print(f"PERGUNTA: {pergunta_texto}")
                
                contexto_recuperado = []
                respostas_modelos = []

                print("   1. Gerando embedding...")
                query_embedding = embed_query(pergunta_texto)

                if query_embedding is None:
                    print("   Falha ao gerar embedding. Pulando esta pergunta.")
                else:
                    print("   2. Recuperando contexto...")
                    contexto_recuperado = retrieve_context(query_embedding, pergunta_texto, NUM_RETRIEVED_DOCS)

                    print(f"   {len(contexto_recuperado)} chunks únicos enviados para o prompt.")

                # O mesmo contexto recuperado é usado nas duas variantes (com e sem instrução)
                for usar_instrucao in VARIANTES_INSTRUCAO:
                    sufixo = "com_instrucao" if usar_instrucao else "sem_instrucao"

                    if query_embedding is None:
                        resposta_rag = f"{ERRO_API}: falha ao gerar embedding"
                    else:
                        print(f"   3. Construindo prompt ({sufixo})...")
                        rag_prompt = construct_rag_prompt(contexto_recuperado, pergunta_texto, usar_instrucao)

                        print("   4. Gerando resposta...")
                        resposta_rag = generate_rag_response_nvidia(rag_prompt)

                    print(f"\n>> Resposta do Modelo (RAG, {sufixo}):")
                    print(textwrap.fill(resposta_rag, width=100))
                    print("="*80)

                    respostas_modelos.append({
                        "modelo": f"{NVIDIA_MODEL}_com_rag_{sufixo}",
                        "modelo_base": NVIDIA_MODEL,
                        "rag": True,
                        "instrucao": usar_instrucao,
                        "resposta_completa": resposta_rag
                    })

                resultado_item = {
                    "linha_excel": int(index),
                    "pergunta": pergunta_texto,
                    "coluna_pergunta": coluna_pergunta,
                    "resposta_verdadeira": str(resposta_verdadeira),
                    "coluna_resposta": coluna_resposta,
                    "contexto_recuperado": contexto_recuperado,
                    "respostas_modelos": respostas_modelos
                }

                todos_os_resultados.append(resultado_item)

                if len(todos_os_resultados) % 20 == 0:
                    salvar_json(todos_os_resultados, ARQUIVO_SAIDA)

    except KeyboardInterrupt:
        print("\n\n!!! INTERRUPÇÃO MANUAL DETECTADA !!!")
        print("Salvando o progresso antes de sair...")
    
    except Exception as e:
        print(f"\n\n!!! ERRO INESPERADO NO LOOP PRINCIPAL: {e} !!!")
        print("Salvando o progresso antes de sair...")

    finally:
        print(f"\nProcessamento interrompido ou concluído. Salvando resultados finais...")
        erros = sum(eh_erro_api(m.get("resposta_completa")) for r in todos_os_resultados for m in r["respostas_modelos"])
        if erros:
            print(f"ATENÇÃO: {erros} respostas ficaram com erro de API. Rode esta célula novamente para refazê-las.")
        salvar_json(todos_os_resultados, ARQUIVO_SAIDA)
        print("Script encerrado.")

if __name__ == "__main__":
    processar_planilha()

In [ ]:
#### COM GOOGLE GEMINI API ###

import os
import pandas as pd
import json
import ollama
import chromadb
# from openai import OpenAI # Não é mais necessário para a inferência
from google import genai # ADICIONADO
from google.genai import types # ADICIONADO
from tqdm import tqdm
import textwrap
import re
import time

# --- CONFIGS NOVAS (GEMINI) ---
GEMINI_API_KEY = "SUA_API_AQUI" # Chave fornecida
GEMINI_MODEL = "gemini-2.5-flash"

DB_PATH = "vector_db_v3"
COLLECTION_NAME = "rag_datasheets_v3"
EMBEDDING_MODEL = "qwen3-embedding"
NUM_RETRIEVED_DOCS = 5 # QUANTIDADE CHUNKS A SEREM RECUPERADOS

# --- NOVAS CONFIGURAÇÕES ---
# Nome do arquivo de entrada (o CSV que você anexou)
ARQUIVO_ENTRADA = "microcontroladores-populares.xlsx"
# Nome do arquivo JSON de saída (Atualizado para refletir o novo modelo)
ARQUIVO_SAIDA = "resultados_rag_gemini-2.5-flash_v3.json"

# Variantes do prompt: True = com instrução ("Se não souber a resposta, retorne 'Não sei'."), False = sem instrução
VARIANTES_INSTRUCAO = [True, False]

# --- TRATAMENTO DE ERROS DE API ---
# Chamadas que falham são repetidas com espera crescente. Se todas as tentativas falharem, a resposta é
# gravada com o prefixo ERRO_API: ela não é enviada para extração, é contada à parte na avaliação
# e é refeita automaticamente na próxima execução do script.
ERRO_API = "erro_api"
TENTATIVAS_API = 3
ESPERA_INICIAL_S = 5  # dobra a cada nova tentativa (5s, 10s, ...)

def eh_erro_api(texto):
    return str(texto).startswith(ERRO_API)

def chamar_com_retentativas(chamada, descricao):
    """Executa 'chamada' (função sem argumentos que retorna o texto da resposta ou levanta exceção).
    Retorna o texto, ou f"{ERRO_API}: <motivo>" se todas as tentativas falharem."""
    for tentativa in range(1, TENTATIVAS_API + 1):
        try:
            return chamada()
        except Exception as e:
            print(f"   [{descricao}] Falha na tentativa {tentativa}/{TENTATIVAS_API}: {e}")
            if tentativa < TENTATIVAS_API:
                time.sleep(ESPERA_INICIAL_S * 2 ** (tentativa - 1))
            else:
                return f"{ERRO_API}: {e}"

# Mapeamento das colunas de pergunta para as colunas de resposta (ground truth)
MAPEAMENTO_PERGUNTAS = {
    "PERGUNTA01": "Flash Memory",
    "PERGUNTA02": "Speed",
    "PERGUNTA03": "Number of I/O",
    "PERGUNTA04": "CommCAN",
    "PERGUNTA05": "CommI2C",
    "PERGUNTA06": "CommEthernet"
}
# --- FIM DAS NOVAS CONFIGURAÇÕES ---


# 2. INICIALIZAÇÃO DOS CLIENTES 
print("--- Iniciando o Processo de Inferência com RAG ---")
print("Usando embeddings do Ollama e inferência do Google Gemini.") # Atualizado

try:

    # --- Cliente GEMINI ---
    client_gemini = genai.Client(api_key=GEMINI_API_KEY)
    
    # --- Cliente Chroma ---
    client_chroma = chromadb.PersistentClient(path=DB_PATH)
    collection = client_chroma.get_collection(name=COLLECTION_NAME)
    
    print(f"Clientes inicializados (Gemini e Chroma). Conectado à coleção '{COLLECTION_NAME}' com {collection.count()} documentos.") # Atualizado
except Exception as e:
    print(f"Erro ao inicializar clientes: {e}")
    raise

# 3. FUNÇÕES DO WORKFLOW RAG

def extract_model_from_query(query: str) -> str | None:

    match = re.search(r'\b(PIC|AT\w*|STM32|LPC)\d+\w*\b', query, re.IGNORECASE)
    return match.group(0) if match else None

def embed_query(query_text): 

    try:
        response = ollama.embeddings(
            model=EMBEDDING_MODEL,
            prompt=query_text
        )
        return response['embedding']
    except Exception as e:
        print(f"Erro ao gerar embedding com Ollama: {e}")
        return None

def deduplicate_results(results_list: list[dict]) -> list[dict]:
   
    """
    Recebe uma lista de dicionários e remove duplicatas com base no 'chunk_content'.
    """
    seen_content = set()
    deduplicated_list = []
    for item in results_list:
        content = item['chunk_content']
        if content not in seen_content:
            seen_content.add(content)
            deduplicated_list.append(item)
    return deduplicated_list

def retrieve_context(query_embedding, query_text, n_results=NUM_RETRIEVED_DOCS):
   
    """
    Recupera chunks, de-duplica e filtra.
    Retorna uma lista de dicionários: [{'chunk_content': '...', 'source_file': '...'}, ...]
    """
    model_in_query = extract_model_from_query(query_text)

    # --- Lógica de busca base ---
    query_params = {
        "query_embeddings": [query_embedding],
        # Aumentamos o N de candidatos para ter mais chance de achar únicos
        "n_results": 20, 
        "include": ["metadatas", "documents"]
    }

    if not model_in_query:
        print("    Nenhum modelo específico detectado. Buscando em todo o banco de dados.")
        try:
            results = collection.query(**query_params)
        except Exception as e:
            print(f"    ERRO ao consultar o ChromaDB: {e}")
            return []
            
        if not results or not results['documents']:
            return []
        
        # Empacota os resultados
        packaged_results = []
        for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
            source_filename = meta.get('source', 'Fonte Desconhecida')
            packaged_results.append({
                "chunk_content": doc,
                "source_file": source_filename
            })
        
        # <<< APLICA A ESTRATÉGIA AQUI >>>
        deduplicated_results = deduplicate_results(packaged_results)
        
        print(f"    Recuperados {len(packaged_results)} chunks, {len(deduplicated_results)} são únicos.")
        # Retorna apenas o N desejado de chunks únicos
        return deduplicated_results[:n_results]

    # --- LÓGICA DE PÓS-FILTRAGEM ---
    print(f"    Modelo '{model_in_query}' detectado. Iniciando busca com pós-filtragem.")
    
    try:
        results = collection.query(**query_params)
    except Exception as e:
        print(f"    ERRO ao consultar o ChromaDB: {e}")
        return []

    if not results or not results['documents']:
        return []

    # Filtra os resultados em Python
    filtered_documents_with_meta = []
    for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
        if meta and 'models' in meta:
            models_metadata = meta['models']
            if isinstance(models_metadata, str) and model_in_query in models_metadata:
                source_filename = meta.get('source', 'Fonte Desconhecida')
                filtered_documents_with_meta.append({
                    "chunk_content": doc,
                    "source_file": source_filename
                })
    
    # <<< APLICA A ESTRATÉGIA AQUI >>>
    deduplicated_results = deduplicate_results(filtered_documents_with_meta)
    print(f"    Filtrados {len(filtered_documents_with_meta)} chunks, {len(deduplicated_results)} são únicos.")

    # Se a filtragem não encontrar nada, usa o fallback
    if not deduplicated_results:
        print(f"    AVISO: Nenhum chunk único encontrado para '{model_in_query}'. Usando fallback.")
        
        fallback_documents = []
        for doc, meta in zip(results['documents'][0], results['metadatas'][0]):
            source_filename = meta.get('source', 'Fonte Desconhecida')
            fallback_documents.append({
                "chunk_content": doc,
                "source_file": source_filename
            })
        
        # De-duplica o fallback também
        deduplicated_fallback = deduplicate_results(fallback_documents)
        return deduplicated_fallback[:n_results]

    return deduplicated_results[:n_results]

def construct_rag_prompt(context_list_of_dicts, question, usar_instrucao=True):
    # (Sem alterações - esta função já constrói o prompt RAG que precisamos)
    """Constrói o prompt final para o LLM a partir da lista de dicionários."""
    
    # Extrai apenas o conteúdo do chunk para o prompt
    chunk_contents = [item['chunk_content'] for item in context_list_of_dicts]
    context_str = "\n\n".join(chunk_contents)
    
    instrucao = " Se não souber a resposta, retorne 'Não sei'." if usar_instrucao else ""
    prompt = (
        "Responda a pergunta \n"
        "Utilize as informações fornecidas no contexto para formular sua resposta."
        f"{instrucao}"
        "\n\n--- CONTEXTO ---\n"
        f"{context_str}"
        "\n\n--- PERGUNTA ---\n"
        f"{question}"
    )
    return prompt


# --- NOVA FUNÇÃO PARA O GEMINI ---
def generate_rag_response_gemini(prompt):
    """Envia o prompt para a API do Google Gemini, usando client.models.generate_content."""
    def chamada():
        print("    Enviando para a API Gemini...")
        response = client_gemini.models.generate_content(
            model=GEMINI_MODEL,
            contents=prompt, # O prompt RAG completo
            config=types.GenerateContentConfig(
            thinking_config=types.ThinkingConfig(thinking_budget=0)
            )
        )
        print("    Resposta recebida.")
        if not response.text:
            raise ValueError("a API não retornou 'text'")
        return response.text.strip()

    return chamar_com_retentativas(chamada, "Gemini")


def salvar_json(data, filename):
   
    """Função auxiliar para salvar o JSON."""
    try:
        with open(filename, 'w', encoding='utf-8') as f:
            json.dump(data, f, ensure_ascii=False, indent=4)
        print(f"\n[Progresso salvo em '{filename}']")
    except Exception as e:
        print(f"\n[ERRO AO SALVAR JSON: {e}]")

def processar_planilha():
    todos_os_resultados = []
    
    try:
        if os.path.exists(ARQUIVO_SAIDA):
            with open(ARQUIVO_SAIDA, 'r', encoding='utf-8') as f:
                todos_os_resultados = json.load(f)
            print(f"Carregados {len(todos_os_resultados)} resultados de execuções anteriores.")
    except Exception as e:
        print(f"Não foi possível carregar resultados anteriores: {e}")
        todos_os_resultados = []

    # Perguntas com erro de API são descartadas para serem refeitas nesta execução
    validos = [r for r in todos_os_resultados
               if not any(eh_erro_api(m.get("resposta_completa")) for m in r.get("respostas_modelos", []))]
    if len(validos) < len(todos_os_resultados):
        print(f"{len(todos_os_resultados) - len(validos)} perguntas com erro de API serão refeitas.")
    todos_os_resultados = validos

    # Perguntas já respondidas em execuções anteriores (permite retomar sem duplicar resultados)
    ja_processadas = {(r.get("linha_excel"), r.get("coluna_pergunta")) for r in todos_os_resultados}

    try:
        df = pd.read_excel(ARQUIVO_ENTRADA)
        print(f"Arquivo '{ARQUIVO_ENTRADA}' carregado. {len(df)} linhas para processar.")
    except FileNotFoundError:
        print(f"ERRO: Arquivo de entrada não encontrado: {ARQUIVO_ENTRADA}")
        return
    except Exception as e:
        print(f"Erro ao ler o arquivo CSV: {e}")
        return

    try:
        print("Iniciando processamento das perguntas...")
        for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Processando Linhas"):
            
            for coluna_pergunta, coluna_resposta in MAPEAMENTO_PERGUNTAS.items():
                
                pergunta_texto = row[coluna_pergunta]
                resposta_verdadeira = row[coluna_resposta]
                
                if not isinstance(pergunta_texto, str) or pd.isna(pergunta_texto):
                    continue

                if (int(index), coluna_pergunta) in ja_processadas:
                    continue

                print("\n" + "="*80)
                print(f"Processando: [Linha {index + 2}] - [{coluna_pergunta}]")
                print(f"PERGUNTA: {pergunta_texto}")
                
                contexto_recuperado = []
                respostas_modelos = []

                print("    1. Gerando embedding...")
                query_embedding = embed_query(pergunta_texto)

                if query_embedding is None:
                    print("    Falha ao gerar embedding. Pulando esta pergunta.")
                else:
                    print("    2. Recuperando contexto...")
                    contexto_recuperado = retrieve_context(query_embedding, pergunta_texto, NUM_RETRIEVED_DOCS)

                    print(f"    {len(contexto_recuperado)} chunks únicos enviados para o prompt.")

                # O mesmo contexto recuperado é usado nas duas variantes (com e sem instrução)
                for usar_instrucao in VARIANTES_INSTRUCAO:
                    sufixo = "com_instrucao" if usar_instrucao else "sem_instrucao"

                    if query_embedding is None:
                        resposta_rag = f"{ERRO_API}: falha ao gerar embedding"
                    else:
                        print(f"    3. Construindo prompt ({sufixo})...")
                        rag_prompt = construct_rag_prompt(contexto_recuperado, pergunta_texto, usar_instrucao)

                        print("    4. Gerando resposta...")
                        resposta_rag = generate_rag_response_gemini(rag_prompt)
                        time.sleep(1)  # Pequena pausa para evitar rate limits

                    print(f"\n>> Resposta do Modelo (RAG, {sufixo}):")
                    print(textwrap.fill(resposta_rag, width=100))
                    print("="*80)

                    respostas_modelos.append({
                        "modelo": f"{GEMINI_MODEL}_com_rag_{sufixo}",
                        "modelo_base": GEMINI_MODEL,
                        "rag": True,
                        "instrucao": usar_instrucao,
                        "resposta_completa": resposta_rag
                    })

                resultado_item = {
                    "linha_excel": int(index),
                    "pergunta": pergunta_texto,
                    "coluna_pergunta": coluna_pergunta,
                    "resposta_verdadeira": str(resposta_verdadeira),
                    "coluna_resposta": coluna_resposta,
                    "contexto_recuperado": contexto_recuperado,
                    "respostas_modelos": respostas_modelos
                }

                todos_os_resultados.append(resultado_item)

                if len(todos_os_resultados) % 20 == 0:
                    salvar_json(todos_os_resultados, ARQUIVO_SAIDA)

    except KeyboardInterrupt:
        print("\n\n!!! INTERRUPÇÃO MANUAL DETECTADA !!!")
        print("Salvando o progresso antes de sair...")
    
    except Exception as e:
        print(f"\n\n!!! ERRO INESPERADO NO LOOP PRINCIPAL: {e} !!!")
        print("Salvando o progresso antes de sair...")

    finally:
        print(f"\nProcessamento interrompido ou concluído. Salvando resultados finais...")
        erros = sum(eh_erro_api(m.get("resposta_completa")) for r in todos_os_resultados for m in r["respostas_modelos"])
        if erros:
            print(f"ATENÇÃO: {erros} respostas ficaram com erro de API. Rode esta célula novamente para refazê-las.")
        salvar_json(todos_os_resultados, ARQUIVO_SAIDA)
        print("Script encerrado.")

if __name__ == "__main__":
    processar_planilha()